# 02 · Tool Calling in a Loop

Phase 1 covered a *single* function call. An agent calls tools **repeatedly**, using each result to decide the next move. Here we look under the hood: the model returns structured `tool_calls`, we execute them, feed the results back, and continue.

> Runs on Ollama. One cell optionally uses OpenAI — skips if no key.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'agents_lab').exists())
sys.path.insert(0, str(root))

First, the raw mechanic: one `chat` call with tool schemas returns a normalized `tool_calls` list of `{'name', 'arguments'}` — the same shape across Ollama, OpenAI, and Anthropic.

In [ ]:
from study_buddy import get_provider
from agents_lab import default_tools

tools = default_tools()
llm = get_provider('ollama')
resp = llm.chat(
    [{'role': 'user', 'content': 'Use a tool to compute 144 / 12.'}],
    tools=tools.schemas,
    temperature=0.0,
)
print('tool_calls:', resp.tool_calls)

Now the full loop via `Agent`. A multi-part task forces *several* tool calls in sequence — each observation informs the next.

In [ ]:
from agents_lab import Agent, Tracer

agent = Agent(provider='ollama', tools=tools, max_steps=6)
tracer = Tracer()
res = agent.run('Compute (12 + 8) * 3, then subtract 15 from the result.', tracer=tracer)
print(tracer.pretty())
print('\nANSWER:', res.answer)

The same loop runs on any provider. Swap one string to route through OpenAI (skips if no key).

In [ ]:
try:
    cloud = Agent(provider='openai', model='gpt-4o-mini', tools=tools)
    print(cloud.run('What is 2 to the power 10?').answer)
except Exception as e:
    print('openai skipped:', type(e).__name__, '-', e)

**Takeaway:** tool calling *in a loop* is what separates an agent from a one-shot function call — the model keeps acting until the task is done.

Next → `03_tool_design.ipynb`